<a href="https://colab.research.google.com/github/Shivani-Sasikumar/Synthetic-Medical-Document-Generation-for-OCR-Training/blob/main/Synthethic_MedicalForms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

# Install required packages
!apt-get install -y fonts-noto fonts-dejavu fonts-liberation
!pip install faker reportlab pillow requests PyPDF2

import requests
import os
from faker import Faker
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from PIL import Image, ImageDraw, ImageFont
import random
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from io import BytesIO
import io
from reportlab.lib.colors import black






Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-liberation is already the newest version (1:1.07.4-11).
fonts-dejavu is already the newest version (2.37-2build1).
fonts-noto is already the newest version (20201225-1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [14]:
def find_coordinates():
    """Helper function to find exact coordinates by drawing a grid"""
    packet = io.BytesIO()
    can = canvas.Canvas(packet, pagesize=letter)
    can.setFont('Helvetica', 6)
    can.setFillColor(black)

    # Draw coordinate grid
    for x in range(0, 600, 20):
        for y in range(0, 800, 20):
            can.drawString(x, y, f"{x},{y}")

    # Mark some key positions
    can.setFont('Helvetica', 10)
    can.setFillColor(black)
    key_positions = [
        (100, 700, "Top Left"),
        (300, 700, "Top Middle"),
        (500, 700, "Top Right"),
        (100, 500, "Middle Left"),
        (300, 500, "Center"),
        (500, 500, "Middle Right"),
        (100, 300, "Bottom Left"),
        (300, 300, "Bottom Middle"),
        (500, 300, "Bottom Right")
    ]

    for x, y, label in key_positions:
        can.drawString(x, y, f"{label} ({x},{y})")

    can.save()

    # Merge with original
    packet.seek(0)
    overlay_pdf = PdfReader(packet)
    original_pdf = PdfReader(open('/medical_start_form.pdf', 'rb'))

    output_pdf = PdfWriter()
    page = original_pdf.pages[0]
    page.merge_page(overlay_pdf.pages[0])
    output_pdf.add_page(page)

    with open('coordinate_finder.pdf', 'wb') as output_file:
        output_pdf.write(output_file)

    print("Coordinate finder PDF created! Use it to find exact positions.")

# Uncomment to create coordinate finder
find_coordinates()

Coordinate finder PDF created! Use it to find exact positions.


In [20]:
##Without edge cases:

import os
import random
from pathlib import Path

import io
from faker import Faker
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import black

def create_colab_folder(folder_name="Medical_Forms_Output"):
    """Create a folder in Colab environment or return existing one"""
    folder_path = f"/content/{folder_name}"

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f" Created folder in Colab: {folder_path}")
    else:
        print(f" Using existing folder in Colab: {folder_path}")

    return folder_path

def get_all_fonts_from_directory():
    """Scan the fonts directory and return all valid font files"""
    fonts_dir = '/fonts'
    font_styles = []

    if not os.path.exists(fonts_dir):
        print(f" Fonts directory '{fonts_dir}' not found!")
        return font_styles

    # Supported font file extensions
    font_extensions = ['.ttf', '.otf', '.TTF', '.OTF']

    print(f" Scanning '{fonts_dir}' directory for fonts...")

    for font_file in os.listdir(fonts_dir):
        if any(font_file.endswith(ext) for ext in font_extensions):
            font_path = os.path.join(fonts_dir, font_file)
            font_name = os.path.splitext(font_file)[0]  # Remove file extension

            font_styles.append({
                'name': font_name,
                'file': font_path,
                'type': 'custom'
            })
            print(f"    Found: {font_name}")

    # If no custom fonts found, use built-in fonts as fallback
    if not font_styles:
        print("   No custom fonts found, using built-in fonts")
        font_styles.extend([
            {'name': 'Helvetica', 'type': 'builtin'},
            {'name': 'Helvetica-Bold', 'type': 'builtin'},
            {'name': 'Times-Roman', 'type': 'builtin'},
            {'name': 'Courier', 'type': 'builtin'},
            {'name': 'Helvetica-Oblique', 'type': 'builtin'}
        ])

    print(f" Total fonts found: {len(font_styles)}")
    return font_styles

def generate_medical_forms_with_auto_fonts():
    fake = Faker()

    # Create folder in Colab environment
    output_folder = create_colab_folder("Medical_Forms_Output")

    # Automatically get all fonts from the directory
    font_styles = get_all_fonts_from_directory()

    if not font_styles:
        print(" No fonts available. Exiting.")
        return

    print(f" Generating {len(font_styles)} medical forms with auto-detected fonts...")

    generated_files = []

    for i, font_style in enumerate(font_styles, 1):
        # Create filename with index (001, 002, etc.)
        index_str = str(i).zfill(3)
        filename = f'medical_form_{index_str}_{font_style["name"].lower()}.pdf'
        output_filename = os.path.join(output_folder, filename)

        print(f"\n [{index_str}] Generating with font: {font_style['name']}")

        try:
            # Generate fake data
            fake_data = {
                'patient_name': f"{fake.first_name()} {fake.last_name()}",
                'patient_dob': fake.date_of_birth(minimum_age=5, maximum_age=80).strftime('%m/%d/%Y'),
                'address': fake.street_address(),
                'city': fake.city(),
                'state': fake.state_abbr(),
                'zip': fake.zipcode(),
                'primary_contact': fake.name(),
                'relationship': random.choice(['Parent', 'Guardian', 'Spouse']),
                'email': fake.email(),
                'phone': fake.phone_number(),

                'primary_insurance': random.choice(['Aetna', 'Blue Cross', 'United Healthcare', 'Cigna']),
                'primary_id': fake.bothify('??#######').upper(),
                'primary_group': fake.bothify('GRP###').upper(),
                'primary_phone': fake.phone_number(),
                'policy_holder': fake.name(),
                'policy_relationship': random.choice(['Self', 'Spouse', 'Parent']),

                'secondary_insurance': random.choice(['Medicare', 'Medicaid']),
                'secondary_id': fake.bothify('??#######').upper(),
                'secondary_group': fake.bothify('GRP###').upper(),
                'secondary_phone': fake.phone_number(),
                'secondary_policy_holder': fake.name(),
                'secondary_relationship': random.choice(['Self', 'Spouse', 'Parent']),

                'physician_name': f"Dr. {fake.first_name()} {fake.last_name()}",
                'facility_name': f"{fake.company()} Medical Center",
                'physician_address': fake.street_address(),
                'suite': f"Suite {random.randint(100, 999)}",
                'physician_city': fake.city(),
                'physician_state': fake.state_abbr(),
                'physician_zip': fake.zipcode(),
                'npi': fake.numerify('##########'),
                'tax_id': fake.numerify('#########'),
                'office_contact': fake.name(),
                'physician_phone': fake.phone_number(),
                'physician_fax': fake.phone_number(),
                'physician_email': fake.email(),

                'physician_print_name': f"Dr. {fake.last_name()}",
                'physician_signature': f"Dr. {fake.last_name()}",
                'physician_date': fake.date_this_year().strftime('%m/%d/%Y'),
                'patient_signature': fake.name(),
                'patient_date': fake.date_this_year().strftime('%m/%d/%Y'),
                'patient_print_name': fake.name(),
                'patient_relationship': random.choice(['Self', 'Parent', 'Guardian'])
            }

           # Your coordinates
            field_coords = {
                'patient_name': (130, 650),
                'patient_dob': (480, 650),
                'address': (80, 637),
                'city': (340, 637),
                'state': (452, 637),
                'zip': (502, 637),
                'primary_contact': (120, 620),
                'relationship': (380, 620),
                'email': (60, 605),
                'phone': (370, 605),

                'primary_insurance': (70, 545),
                'primary_id': (240, 545),
                'primary_group': (330, 545),
                'primary_phone': (450, 545),
                'policy_holder': (82, 530),
                'policy_relationship': (382, 530),

                'secondary_insurance': (83, 515),
                'secondary_id': (240, 515),
                'secondary_group': (340, 515),
                'secondary_phone': (450, 510),
                'secondary_policy_holder': (85, 500),
                'secondary_relationship': (400, 500),

                'physician_name': (100, 440),
                'facility_name': (360, 440),
                'physician_address': (80, 425),
                'suite': (245, 425),
                'physician_city': (320, 425),
                'physician_state': (450, 425),
                'physician_zip': (500, 425),
                'npi': (60, 425),
                'tax_id': (280, 405),
                'office_contact': (450, 405),
                'physician_phone': (70, 390),
                'physician_fax': (190, 390),
                'physician_email': (330, 390),

                'physician_print_name': (140, 219),
                'physician_signature': (130, 200),
                'physician_date': (510, 200),

                'patient_signature': (290, 100),
                'patient_date': (520, 100),
                'patient_print_name': (260, 80),
                'patient_relationship': (120, 60),
            }

            # Create overlay
            packet = io.BytesIO()
            can = canvas.Canvas(packet, pagesize=letter)

            # Set font based on type
            if font_style['type'] == 'builtin':
                can.setFont(font_style['name'], 9)
            else:
                try:
                    pdfmetrics.registerFont(TTFont(font_style['name'], font_style['file']))
                    # Adjust font size based on font characteristics
                    font_size = 11
                    script_fonts = ['GreatVibes', 'DancingScript', 'Caveat', 'Pacifico', 'BrushScript', 'LucidaCalligraphy']
                    handwriting_fonts = ['Schoolbell', 'ShadowsIntoLight', 'IndieFlower', 'ComicSans', 'Chalkboard']

                    if any(script in font_style['name'] for script in script_fonts):
                        font_size = 12
                    elif any(hand in font_style['name'] for hand in handwriting_fonts):
                        font_size = 13

                    can.setFont(font_style['name'], font_size)
                except Exception as e:
                    print(f"   Custom font failed, using Helvetica as fallback: {str(e)}")
                    can.setFont('Helvetica', 11)

            can.setFillColor(black)

            # Draw fields
            for field, (x, y) in field_coords.items():
                if field in fake_data:
                    text = str(fake_data[field])
                    if field.startswith('secondary_') and not text.strip():
                        continue
                    if len(text) > 25:
                        text = text[:22] + "..."
                    can.drawString(x, y, text)

            can.save()

            # Merge with original PDF
            packet.seek(0)
            overlay_pdf = PdfReader(packet)
            original_pdf = PdfReader(open('/medical_start_form.pdf', 'rb'))

            output_pdf = PdfWriter()

            for page_num in range(len(original_pdf.pages)):
                page = original_pdf.pages[page_num]
                if page_num == 0:
                    page.merge_page(overlay_pdf.pages[0])
                output_pdf.add_page(page)

            # Save to Colab folder
            with open(output_filename, 'wb') as output_file:
                output_pdf.write(output_file)

            # Check if file was created and get its size
            if os.path.exists(output_filename):
                file_size = os.path.getsize(output_filename)
                generated_files.append({
                    'filename': filename,
                    'full_path': output_filename,
                    'size': file_size,
                    'font': font_style['name']
                })
                print(f" Generated: {filename} ({file_size:,} bytes)")
                print(f"    Saved to: {output_filename}")
            else:
                print(f" File not created: {output_filename}")

        except Exception as e:
            print(f"Failed to generate {filename}: {str(e)}")

    # Summary
    print(f"\n PDF Generation Summary:")
    print(f"Total files generated: {len(generated_files)}")
    print(f" Files saved in: {output_folder}")
    print(f"\n Generated Files in Colab:")

    for file_info in generated_files:
        size_mb = file_info['size'] / 1024 / 1024
        print(f"    {file_info['filename']} - {file_info['font']} - {size_mb:.2f} MB")

    # Show how to download files from Colab
    print(f"\n To download all files from Colab, run:")
    print(f"   from google.colab import files")
    print(f"   import shutil")
    print(f"   shutil.make_archive('medical_forms', 'zip', '{output_folder}')")
    print(f"   files.download('medical_forms.zip')")

# Run the function
print(" Starting Medical Form PDF Generation in Colab...")
generate_medical_forms_with_auto_fonts()

# Optional: List all files in the output folder
print(f"\n Contents of output folder:")
output_folder = "/content/Medical_Forms_Output"
if os.path.exists(output_folder):
    for file in os.listdir(output_folder):
        if file.endswith('.pdf'):
            file_path = os.path.join(output_folder, file)
            file_size = os.path.getsize(file_path)
            print(f"    {file} - {file_size:,} bytes")

 Starting Medical Form PDF Generation in Colab...
 Using existing folder in Colab: /content/Medical_Forms_Output
 Scanning '/fonts' directory for fonts...
    Found: lomesty.regular
    Found: Akeylah__s_Handwriting
    Found: Elizany
    Found: Berthusen Demo
    Found: Abhyaksa FREE
    Found: a Theme for murder
    Found: Hello Bunda Demo
 Total fonts found: 7
 Generating 7 medical forms with auto-detected fonts...

 [001] Generating with font: lomesty.regular
 Generated: medical_form_001_lomesty.regular.pdf (182,855 bytes)
    Saved to: /content/Medical_Forms_Output/medical_form_001_lomesty.regular.pdf

 [002] Generating with font: Akeylah__s_Handwriting
 Generated: medical_form_002_akeylah__s_handwriting.pdf (167,828 bytes)
    Saved to: /content/Medical_Forms_Output/medical_form_002_akeylah__s_handwriting.pdf

 [003] Generating with font: Elizany
 Generated: medical_form_003_elizany.pdf (175,183 bytes)
    Saved to: /content/Medical_Forms_Output/medical_form_003_elizany.pdf

 [00

In [19]:
#With edge cases:

import os
import random
from pathlib import Path
import random
import io
from faker import Faker
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import black

def create_colab_folder(folder_name="Medical_Forms_OutputV01"):
    """Create a folder in Colab environment or return existing one"""
    folder_path = f"/content/{folder_name}"

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f" Created folder in Colab: {folder_path}")
    else:
        print(f" Using existing folder in Colab: {folder_path}")

    return folder_path

def get_all_fonts_from_directory():
    """Scan the fonts directory and return all valid font files"""
    fonts_dir = '/fonts'
    font_styles = []

    if not os.path.exists(fonts_dir):
        print(f" Fonts directory '{fonts_dir}' not found!")
        return font_styles

    font_size = 14
    font_extensions = ['.ttf', '.otf', '.TTF', '.OTF']

    print(f" Scanning '{fonts_dir}' directory for fonts...")

    for font_file in os.listdir(fonts_dir):
        if any(font_file.endswith(ext) for ext in font_extensions):
            font_path = os.path.join(fonts_dir, font_file)
            font_name = os.path.splitext(font_file)[0]  # Remove file extension

            font_styles.append({
                'name': font_name,
                'file': font_path,
                'type': 'custom'
            })
            print(f"    Found: {font_name}")

    # If no custom fonts found, use built-in fonts as fallback
    if not font_styles:
        print("   No custom fonts found, using built-in fonts")
        font_styles.extend([
            {'name': 'Helvetica', 'type': 'builtin'},
            {'name': 'Helvetica-Bold', 'type': 'builtin'},
            {'name': 'Times-Roman', 'type': 'builtin'},
            {'name': 'Courier', 'type': 'builtin'},
            {'name': 'Helvetica-Oblique', 'type': 'builtin'}
        ])

    print(f" Total fonts found: {len(font_styles)}")
    return font_styles

def add_messy_effects(canvas, x, y, text, font_size):
    """Add messy writing effects like scribbles, corrections, etc."""
    # Randomly decide if we should add messy effects (30% chance)
    if random.random() < 0.3:
        effect_type = random.choice(['scribble', 'correction', 'smudge', 'overlap', 'wavy'])

        if effect_type == 'scribble':
            # Add random scribble lines near the text
            for _ in range(random.randint(1, 3)):
                scribble_x = x + random.randint(-5, len(text) * font_size // 2)
                scribble_y = y + random.randint(-3, 3)
                scribble_length = random.randint(10, 30)
                canvas.line(scribble_x, scribble_y, scribble_x + scribble_length, scribble_y)

        elif effect_type == 'correction':
            # Draw crossed-out text
            canvas.line(x-2, y+2, x + len(text) * font_size // 2, y-2)
            # Then draw the corrected text slightly offset
            return x + 5, y - 2

        elif effect_type == 'overlap':
            # Draw text slightly overlapping
            canvas.setFillColorRGB(0.7, 0.7, 0.7)  # Light gray
            canvas.drawString(x + 1, y - 1, text)
            canvas.setFillColor(black)

    return x, y

def generate_edge_case_data(fake, field_name):
    """Generate edge case data for specific fields"""

    edge_cases = {
        # Messy/incorrect data
        'patient_name': [
            f"{fake.first_name().upper()} {fake.last_name()}",
            f"{fake.first_name()} {fake.last_name().lower()}",
            f"{fake.first_name()[0]}. {fake.last_name()}",
            f"{fake.first_name()} {fake.last_name()} {fake.last_name()}",
            "",  # Missing field
        ],
        'patient_dob': [
            "00/00/0000",
            "13/45/1990",  # Invalid date
            "01-01-1990",  # Wrong format
            "1990/01/01",  # Wrong format
            "",  # Missing
        ],
        'address': [
            "123 Main St",  # Incomplete
            "PO Box 123",
            "N/A",
            "Same as above",
            "",  # Missing
        ],
        'phone': [
            "555-phone",  # Invalid
            "123",  # Too short
            "555-555-5555-5555",  # Too long
            "call me",
            "",  # Missing
        ],
        'email': [
            "invalid-email",
            "name@",
            "@domain.com",
            "no email",
            "",  # Missing
        ],
        'zip': [
            "1234",  # Too short
            "123456",  # Too long
            "ABCDE",  # Letters
            "00000",
            "",  # Missing
        ],
        'primary_id': [
            "N/A",
            "UNKNOWN",
            "123",
            "ID# PENDING",
            "",  # Missing
        ],
        'npi': [
            "123",  # Too short
            "12345678901",  # Too long
            "N/A",
            "PENDING",
            "",  # Missing
        ]
    }

    # 20% chance to use edge case data
    if random.random() < 0.2 and field_name in edge_cases:
        return random.choice(edge_cases[field_name])

    return None

def add_correction_marks(canvas, x, y, field_width,font_size):

    mark_type = random.choice(['arrow','line', 'highlight'])

    if mark_type == 'arrow':
        arrow_x = x - 15
        arrow_y = y + 5
        canvas.line(arrow_x, arrow_y, x, y)
        canvas.line(x, y, x-3, y-3)
        canvas.line(x, y, x-3, y+3)

    elif mark_type == 'highlight':
        canvas.setFillColorRGB(1, 1, 0.7)  # Light yellow
        canvas.rect(x - 2, y - 2, field_width + 4, font_size + 4, fill=1, stroke=0)
        canvas.setFillColor(black)

def generate_medical_forms_with_edge_cases():
    fake = Faker()

    # Create folder in Colab environment
    output_folder = create_colab_folder("Medical_Forms_OutputV01")

    # Automatically get all fonts from the directory
    font_styles = get_all_fonts_from_directory()

    if not font_styles:
        print(" No fonts available. Exiting.")
        return

    print(f" Generating {len(font_styles)} medical forms with edge cases...")

    generated_files = []

    for i, font_style in enumerate(font_styles, 1):
        # Create filename with index (001, 002, etc.)
        index_str = str(i).zfill(3)
        filename = f'medical_form_{index_str}_{font_style["name"].lower()}.pdf'
        output_filename = os.path.join(output_folder, filename)

        print(f"\n [{index_str}] Generating with font: {font_style['name']}")

        try:
            # Generate base fake data
            base_data = {
                'patient_name': f"{fake.first_name()} {fake.last_name()}",
                'patient_dob': fake.date_of_birth(minimum_age=5, maximum_age=80).strftime('%m/%d/%Y'),
                'address': fake.street_address(),
                'city': fake.city(),
                'state': fake.state_abbr(),
                'zip': fake.zipcode(),
                'primary_contact': fake.name(),
                'relationship': random.choice(['Parent', 'Guardian', 'Spouse']),
                'email': fake.email(),
                'phone': fake.phone_number(),

                'primary_insurance': random.choice(['Aetna', 'Blue Cross', 'United Healthcare', 'Cigna']),
                'primary_id': fake.bothify('??#######').upper(),
                'primary_group': fake.bothify('GRP###').upper(),
                'primary_phone': fake.phone_number(),
                'policy_holder': fake.name(),
                'policy_relationship': random.choice(['Self', 'Spouse', 'Parent']),

                'secondary_insurance': random.choice(['Medicare', 'Medicaid', '']),
                'secondary_id': fake.bothify('??#######').upper(),
                'secondary_group': fake.bothify('GRP###').upper(),
                'secondary_phone': fake.phone_number(),
                'secondary_policy_holder': fake.name(),
                'secondary_relationship': random.choice(['Self', 'Spouse', 'Parent']),

                'physician_name': f"Dr. {fake.first_name()} {fake.last_name()}",
                'facility_name': f"{fake.company()} Medical Center",
                'physician_address': fake.street_address(),
                'suite': f"Suite {random.randint(100, 999)}",
                'physician_city': fake.city(),
                'physician_state': fake.state_abbr(),
                'physician_zip': fake.zipcode(),
                'npi': fake.numerify('##########'),
                'tax_id': fake.numerify('#########'),
                'office_contact': fake.name(),
                'physician_phone': fake.phone_number(),
                'physician_fax': fake.phone_number(),
                'physician_email': fake.email(),

                'physician_print_name': f"Dr. {fake.last_name()}",
                'physician_signature': f"Dr. {fake.last_name()}",
                'physician_date': fake.date_this_year().strftime('%m/%d/%Y'),
                'patient_signature': fake.name(),
                'patient_date': fake.date_this_year().strftime('%m/%d/%Y'),
                'patient_print_name': fake.name(),
                'patient_relationship': random.choice(['Self', 'Parent', 'Guardian'])
            }

            # Apply edge cases to the data
            fake_data = {}
            for field, value in base_data.items():
                edge_case_value = generate_edge_case_data(fake, field)
                if edge_case_value is not None:
                    fake_data[field] = edge_case_value
                    print(f"   Edge case applied to {field}: '{edge_case_value}'")
                else:
                    fake_data[field] = value

            # Your coordinates
            field_coords = {
                'patient_name': (130, 650),
                'patient_dob': (480, 650),
                'address': (80, 637),
                'city': (340, 637),
                'state': (452, 637),
                'zip': (502, 637),
                'primary_contact': (120, 620),
                'relationship': (380, 620),
                'email': (60, 605),
                'phone': (370, 605),

                'primary_insurance': (70, 545),
                'primary_id': (240, 545),
                'primary_group': (330, 545),
                'primary_phone': (450, 545),
                'policy_holder': (82, 530),
                'policy_relationship': (382, 530),

                'secondary_insurance': (83, 515),
                'secondary_id': (240, 515),
                'secondary_group': (340, 515),
                'secondary_phone': (450, 510),
                'secondary_policy_holder': (85, 500),
                'secondary_relationship': (400, 500),

                'physician_name': (100, 440),
                'facility_name': (360, 440),
                'physician_address': (80, 425),
                'suite': (245, 425),
                'physician_city': (320, 425),
                'physician_state': (450, 425),
                'physician_zip': (500, 425),
                'npi': (60, 405),
                'tax_id': (280, 405),
                'office_contact': (450, 405),
                'physician_phone': (70, 390),
                'physician_fax': (190, 390),
                'physician_email': (330, 390),

                'physician_print_name': (140, 219),
                'physician_signature': (130, 200),
                'physician_date': (510, 200),

                'patient_signature': (290, 100),
                'patient_date': (520, 100),
                'patient_print_name': (260, 80),
                'patient_relationship': (120, 60),
            }

            # Create overlay
            packet = io.BytesIO()
            can = canvas.Canvas(packet, pagesize=letter)

            # Set font based on type
            font_size = 11
            if font_style['type'] == 'builtin':
                can.setFont(font_style['name'], font_size)
            else:
                try:
                    pdfmetrics.registerFont(TTFont(font_style['name'], font_style['file']))
                    # Adjust font size based on font characteristics
                    script_fonts = ['GreatVibes', 'DancingScript', 'Caveat', 'Pacifico', 'BrushScript', 'LucidaCalligraphy']
                    handwriting_fonts = ['Schoolbell', 'ShadowsIntoLight', 'IndieFlower', 'ComicSans', 'Chalkboard']

                    if any(script in font_style['name'] for script in script_fonts):
                        font_size = 12
                    elif any(hand in font_style['name'] for hand in handwriting_fonts):
                        font_size = 13

                    can.setFont(font_style['name'], font_size)
                except Exception as e:
                    print(f"   Custom font failed, using Helvetica as fallback: {str(e)}")
                    can.setFont('Helvetica', font_size)

            can.setFillColor(black)

            # Draw fields with edge cases
            for field, (x, y) in field_coords.items():
                if field in fake_data:
                    text = str(fake_data[field])



                    # Truncate very long text to fit in fields
                    if len(text) > 25:
                        text = text[:22] + "..."

                    # Apply messy effects
                    new_x, new_y = add_messy_effects(can, x, y, text, font_size)

                    # Draw the text
                    can.drawString(new_x, new_y, text)

                    # Occasionally add correction marks (10% chance)
                    if random.random() < 0.1:
                        field_width = len(text) * font_size // 2
                        add_correction_marks(can, new_x, new_y, field_width,font_size)



            can.save()

            # Merge with original PDF
            packet.seek(0)
            overlay_pdf = PdfReader(packet)
            original_pdf = PdfReader(open('/medical_start_form.pdf', 'rb'))

            output_pdf = PdfWriter()

            for page_num in range(len(original_pdf.pages)):
                page = original_pdf.pages[page_num]
                if page_num == 0:
                    page.merge_page(overlay_pdf.pages[0])
                output_pdf.add_page(page)

            # Save to Colab folder
            with open(output_filename, 'wb') as output_file:
                output_pdf.write(output_file)

            # Check if file was created and get its size
            if os.path.exists(output_filename):
                file_size = os.path.getsize(output_filename)
                generated_files.append({
                    'filename': filename,
                    'full_path': output_filename,
                    'size': file_size,
                    'font': font_style['name']
                })
                print(f" Generated: {filename} ({file_size:,} bytes)")
                print(f"    Saved to: {output_filename}")
            else:
                print(f" File not created: {output_filename}")

        except Exception as e:
            print(f"Failed to generate {filename}: {str(e)}")

    # Summary
    print(f"\nPDF Generation Summary:")
    print(f"Total files generated: {len(generated_files)}")
    print(f" Files saved in: {output_folder}")
    print(f"\n Generated Files in Colab:")

    for file_info in generated_files:
        size_mb = file_info['size'] / 1024 / 1024
        print(f"    {file_info['filename']} - {file_info['font']} - {size_mb:.2f} MB")

    # Show how to download files from Colab
    print(f"\n To download all files from Colab, run:")
    print(f"   from google.colab import files")
    print(f"   import shutil")
    print(f"   shutil.make_archive('medical_forms', 'zip', '{output_folder}')")
    print(f"   files.download('medical_forms.zip')")

# Run the function
print(" Starting Medical Form PDF Generation with Edge Cases...")
generate_medical_forms_with_edge_cases()

# Optional: List all files in the output folder
print(f"\n Contents of output folder:")
output_folder = "/content/Medical_Forms_OutputV01"
if os.path.exists(output_folder):
    for file in os.listdir(output_folder):
        if file.endswith('.pdf'):
            file_path = os.path.join(output_folder, file)
            file_size = os.path.getsize(file_path)
            print(f"    {file} - {file_size:,} bytes")

 Starting Medical Form PDF Generation with Edge Cases...
 Using existing folder in Colab: /content/Medical_Forms_OutputV01
 Scanning '/fonts' directory for fonts...
    Found: lomesty.regular
    Found: Akeylah__s_Handwriting
    Found: Elizany
    Found: Berthusen Demo
    Found: Abhyaksa FREE
    Found: a Theme for murder
    Found: Hello Bunda Demo
 Total fonts found: 7
 Generating 7 medical forms with edge cases...

 [001] Generating with font: lomesty.regular
Failed to generate medical_form_001_lomesty.regular.pdf: [Errno 2] No such file or directory: 'medical_start_form.pdf'

 [002] Generating with font: Akeylah__s_Handwriting
   Edge case applied to npi: 'N/A'
Failed to generate medical_form_002_akeylah__s_handwriting.pdf: [Errno 2] No such file or directory: 'medical_start_form.pdf'

 [003] Generating with font: Elizany
   Edge case applied to address: 'N/A'
   Edge case applied to email: 'invalid-email'
   Edge case applied to primary_id: '123'
   Edge case applied to npi: 'PE